# Starrydata2 family / process / variable audit (Colab version)

Re-runs the Starrydata2 analysis already done for you — useful if you want to check a different
family later, or re-run after Starrydata2 updates its release. No blocked hosts here; this runs
fine in this environment too, but Colab makes it easy to tweak and re-run.

Set `FAMILY` to any tag Starrydata2 uses, e.g. `"Half-Heusler"`, `"Skutterudite"`, `"Bi2Te3"`,
`"PbTe"`, `"SnSe"`, `"Mg2Si"`, `"Clathrate"`, `"Zintl"` — the full list of tags found in this
project is printed in the first cell below.

**Runtime > Run all**, then check the printed summary and download the exported CSVs from the
Files sidebar.


In [ ]:
FAMILY = "Half-Heusler"  # change this and re-run


In [ ]:
import json
import pandas as pd

BASE = "https://github.com/atsumitanaka/starrydata-datasets/releases/latest/download/"

papers = pd.read_csv(BASE + "ThermoelectricMaterials_papers.csv.gz")
samples = pd.read_csv(BASE + "ThermoelectricMaterials_samples.csv.gz")
curves = pd.read_csv(BASE + "ThermoelectricMaterials_curves.csv.gz")

print(f"papers: {len(papers)}, samples: {len(samples)}, curves: {len(curves)}")


In [ ]:
def parse_info(info):
    if not isinstance(info, str) or not info.strip():
        return {}
    try:
        return json.loads(info)
    except Exception:
        return {}

samples["info"] = samples["sample_info"].apply(parse_info)
samples["family_tag"] = samples["info"].apply(lambda d: (d.get("MaterialFamily") or {}).get("category") or None)
samples["fab_cat"] = samples["info"].apply(lambda d: (d.get("FabricationProcess") or {}).get("category") or None)
samples["fab_comment"] = samples["info"].apply(lambda d: (d.get("FabricationProcess") or {}).get("comment") or None)
samples["density"] = samples["info"].apply(lambda d: (d.get("RelativeDensity") or {}).get("extracted") or None)
samples["grainsize"] = samples["info"].apply(lambda d: (d.get("GrainSize") or {}).get("extracted") or None)

print("Available MaterialFamily tags in this dataset:")
print(samples["family_tag"].value_counts().head(40))


In [ ]:
seebeck_ids = set(curves.loc[curves["prop_y"] == "Seebeck coefficient", "sample_id"])
carrier_ids = (
    set(curves.loc[curves["prop_y"].isin(["Carrier concentration", "Hall coefficient"]), "sample_id"])
    | set(curves.loc[curves["prop_x"] == "Carrier concentration", "sample_id"])
)
both_ids = seebeck_ids & carrier_ids

sub = samples[samples["family_tag"] == FAMILY].copy()
sub["has_seebeck"] = sub["sample_id"].isin(seebeck_ids)
sub["has_carrier_conc"] = sub["sample_id"].isin(carrier_ids)

print(f"=== {FAMILY} ===")
print(f"total tagged samples:                    {len(sub)}")
print(f"with a Seebeck curve:                    {sub[\'has_seebeck\'].sum()}")
print(f"with a carrier-concentration/Hall curve: {sub[\'has_carrier_conc\'].sum()}")
print(f"with BOTH (usable for base-eq matching): {(sub[\'has_seebeck\'] & sub[\'has_carrier_conc\']).sum()}")
print(f"with RelativeDensity filled:             {sub[\'density\'].notna().sum()}")
print(f"with GrainSize filled:                   {sub[\'grainsize\'].notna().sum()}")


In [ ]:
both = sub[sub["has_seebeck"] & sub["has_carrier_conc"]]

print("FabricationProcess category counts (within the Seebeck+carrier-conc overlap):")
print(both["fab_cat"].value_counts(dropna=False))
print()
print("Free-text FabricationProcess comments in that overlap (often more specific):")
for c in both["fab_comment"].dropna().unique()[:25]:
    print(" -", c[:150])


In [ ]:
out_all = f"starrydata2_{FAMILY.replace(\' \', \'\').replace(\'-\', \'\').lower()}_all_samples.csv"
out_overlap = f"starrydata2_{FAMILY.replace(\' \', \'\').replace(\'-\', \'\').lower()}_seebeck_plus_carrierconc.csv"
sub.drop(columns=["info"]).to_csv(out_all, index=False)
both.drop(columns=["info"]).to_csv(out_overlap, index=False)
print(f"Wrote: {out_all}, {out_overlap} \u2014 download from the Files sidebar on the left.")
